# ML-03 — Frame Your Lane as an ML Task

## Context

For this FlyRank lane, I frame the problem around identifying content pages that are likely to decline so a content/SEO team can prioritize review and refresh work. The goal is the decision, not the model itself.


## 1. My lane as an ML task (type)

**Task type: Classification.**

The question is: **will this content page be observed as declining?** That is a binary outcome, so classification is the clearest framing. The output supports a practical decision: which pages should the content/SEO team review first.

A wrong call has an asymmetric operational cost: a missed declining page can delay a useful refresh, while reviewing a page that turns out not to be declining mainly costs analyst/editor time.


In [1]:
task_type = "classification"
print(f"Task type: {task_type.title()}")


Task type: Classification


## 2. Target or proxy

**Target: `is_declining_label`.**

The starter pipeline defines this observed label as 1 when `trend_direction == "down"`, and 0 otherwise. This is an observed outcome in the supplied slice rather than a label invented from the model's own prediction.

**Leakage warning:** `trend_direction` and `trend_pct` are label-source columns, so they must not be used as model features. The target is defined from them, which would otherwise let the model see the answer.


In [2]:
target = "is_declining_label"
print(f"Target column: {target}")
print("Positive class: 16,262 / 30,000 (54.2%)")
print("Leakage columns excluded from features: trend_direction, trend_pct")


Target column: is_declining_label
Positive class: 16,262 / 30,000 (54.2%)
Leakage columns excluded from features: trend_direction, trend_pct


## 3. Success metric

**Primary success metric: Recall.**

The decision is to surface pages that need attention. Missing a genuinely declining page is more costly than sending some extra pages for review, so recall is a defensible primary metric for the first version. I would report precision alongside recall in later modeling because editor time is also a real cost, but recall is the main success criterion for this framing.

The metric is defined before training: a better model is one that catches more of the observed declining pages without relying on the leaked label-source columns.


In [3]:
primary_metric = "Recall"
print(f"Primary metric: {primary_metric}")
print("Reason: prioritize catching genuinely declining pages")


Primary metric: Recall
Reason: prioritize catching genuinely declining pages


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one row = one pseudonymized content item/page.**

The starter data contains 30,000 rows and 44 columns. I load the actual CSV and show the page-level records below. IDs are used to identify/group records, not as predictive features.


In [4]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Unit of analysis: one row = one content item/page")
display(df.head(10))


In [5]:
assert df.shape == (30000, 44), f"Unexpected starter shape: {df.shape}"
assert df["content_id"].nunique() == len(df), "Expected one row per content item"
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Unique content_id values: {df['content_id'].nunique():,}")


Rows: 30,000
Columns: 44
Unique content_id values: 30,000


In [6]:
if target not in df.columns:
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target values:")
print(df[target].value_counts().sort_index())


Target values:
0    13738
1    16262
Name: is_declining_label, dtype: int64


## 5. Why ML beats a fixed rule here

A single threshold rule such as `impressions_last_30d < impressions_prev_30d` is easy to write, but it throws away the wider context available for each page. Decline risk can depend on a combination of search demand, clicks, sessions, CTR, search position, content age, update recency, keyword context, and content type. These signals can interact, and their useful relationships do not have to share one hand-written threshold.

ML earns its place if it can learn a repeatable pattern across those observed signals and improve the decision of **which pages the content/SEO team should review first**. If a simple rule performs just as well, then the rule is preferable; ML is not automatically better. For this phase, the claim is therefore **decision-support**, not an assertion that a model will definitely outperform a rule before it is tested.


In [7]:
decision = "prioritize content pages for review/refresh"
actor = "content/SEO team"
claim_level = "decision-support"
print(f"Decision supported: {decision}")
print(f"Who acts: {actor}")
print(f"Claim level: {claim_level}")


Decision supported: prioritize content pages for review/refresh
Who acts: content/SEO team
Claim level: decision-support


## Self-check

- [x] Task type is named and justified: classification.
- [x] Target/proxy is named: `is_declining_label`.
- [x] Success metric is named before modeling: recall.
- [x] Unit of analysis is demonstrated with the real starter dataframe: one row per content item/page.
- [x] The output is tied to a real content action: prioritize review/refresh work.
- [x] The ML-vs-rule argument is conditional and honest: ML must earn its place by improving the decision.
- [x] Label-source leakage columns are explicitly excluded: `trend_direction`, `trend_pct`.
- [x] No client names, URLs, or private queries are included.
